### 1. Imports and shared helpers
Imports Python standard-library modules and the scientific packages used by this notebook: NumPy, pandas, Matplotlib, PyTorch, and scikit-learn metrics.

In [1]:
"""DNN Assignment 2, Problem 1.
"""

from __future__ import annotations

import csv
import random
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Iterable

import re
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score


### 2. Experiment configuration
Sets the chosen recurrent hidden size, the three random seeds, epoch budgets, batch size, learning rate, compute device, and shared results-file path.

In [2]:
# Clarification: students choose their own values. Change these and report them.
HIDDEN_SIZE = 64
SEEDS = (1, 42, 123)
EPOCHS_HAR = 20
EPOCHS_ADDING = 30
BATCH_SIZE = 128
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_PATH = Path("..") / "results.csv"

RESULT_COLUMNS = [
    "problem", "model", "seed", "seq_len", "test_acc", "macro_f1",
    "test_mse", "params", "sec_per_epoch", "val_loss_nats",
    "bits_per_char", "notes",
]

### 3. Select the individual ablation
Uses the last digit of the student ID only to select the assignment’s even/odd Task 5(c) ablation. The hidden size and seeds above are chosen independently.

In [3]:
# Professor clarification: HIDDEN_SIZE and SEEDS above are chosen by you.
# The student ID is used only for Problem 1 Task 5(c) parity ablation.
STUDENT_ID = "2026AJ05005"
LAST_ID_DIGIT = next((int(ch) for ch in reversed(STUDENT_ID) if ch.isdigit()), None)
if LAST_ID_DIGIT is None:
    raise ValueError("STUDENT_ID must contain a digit")
ABLATION_CHOICE = "gradient_clip" if LAST_ID_DIGIT % 2 == 0 else "forget_bias"
print(f"Task 5(c) ablation selected from ID parity: {ABLATION_CHOICE}")


Task 5(c) ablation selected from ID parity: forget_bias


### 4. Reproducibility helper
Defines a helper that seeds Python, NumPy, PyTorch, and available CUDA devices so repeated runs use controlled randomness.

In [4]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


### 5. Save experiment rows
Defines the CSV writer used by the experiments. Each completed run is appended with the common results columns so long runs retain completed results.

In [5]:
def append_result(row: dict, path: Path = RESULTS_PATH) -> None:
    """Append one completed run, creating the required exact-column CSV."""
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_COLUMNS)
        if not exists:
            writer.writeheader()
        writer.writerow({c: row.get(c, "") for c in RESULT_COLUMNS})



### 6. Download and load UCI HAR
Finds an existing dataset or downloads the assignment ZIP into the relative har folder, extracts a possible nested ZIP, and loads the nine sensor channels and labels. It creates a four-subject validation split from the training subjects and calculates normalization statistics using training subjects only.

In [6]:
HAR_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
HAR_DATA_DIR = Path("har")
HAR_CHANNELS = [
    "body_acc_x", "body_acc_y", "body_acc_z", "body_gyro_x",
    "body_gyro_y", "body_gyro_z", "total_acc_x", "total_acc_y", "total_acc_z",
]


def _is_har_root(path: Path) -> bool:
    """Return True when path contains the extracted train and test HAR folders."""
    return (
        (path / "train" / "Inertial Signals").is_dir()
        and (path / "test" / "Inertial Signals").is_dir()
        and (path / "train" / "y_train.txt").is_file()
        and (path / "test" / "y_test.txt").is_file()
    )


def ensure_har_dataset(data_dir: str | Path = HAR_DATA_DIR) -> Path:
    """Find the local UCI HAR dataset or download and extract it into relative folder ./har.

    Handles both a ZIP containing the dataset directly and the nested
    'UCI HAR Dataset.zip' archive mentioned in the assignment instructions.
    """
    data_dir = Path(data_dir)
    candidates = (
        data_dir,
        data_dir / "UCI HAR Dataset",
        Path("UCI HAR Dataset"),
        Path("har") / "UCI HAR Dataset",
        Path("..") / "problem1" / "UCI HAR Dataset",
    )
    for candidate in candidates:
        if _is_har_root(candidate):
            return candidate

    data_dir.mkdir(parents=True, exist_ok=True)
    archive_path = data_dir / "har.zip"
    print(f"Downloading UCI HAR dataset to {archive_path} ...")
    urllib.request.urlretrieve(HAR_URL, archive_path)

    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(data_dir)

    # Some copies of the outer archive contain a second ZIP; extract it as well.
    for nested_zip in data_dir.rglob("UCI HAR Dataset.zip"):
        with zipfile.ZipFile(nested_zip) as archive:
            archive.extractall(data_dir)

    # Locate the actual dataset directory, including if the ZIP added an outer folder.
    for train_dir in data_dir.rglob("train"):
        candidate = train_dir.parent
        if _is_har_root(candidate):
            print(f"HAR dataset ready at: {candidate}")
            return candidate
    raise FileNotFoundError(
        f"Downloaded {archive_path}, but could not find train/test HAR folders. "
        "Check the archive contents and extraction permissions."
    )


def load_har(root: str | Path = HAR_DATA_DIR, seed: int = 0):
    """Load HAR with a seeded four-subject validation split and train-only statistics.

    Returns normalized float arrays and labels 0..5. Each sequence has 128
    time steps and 9 channels in the assignment's fixed channel order.
    """
    root = ensure_har_dataset(root)

    def split(name: str):
        base = root / name
        x = np.stack([
            np.loadtxt(base / "Inertial Signals" / f"{channel}_{name}.txt")
            for channel in HAR_CHANNELS
        ], axis=-1).astype(np.float32)
        y = np.loadtxt(base / f"y_{name}.txt", dtype=np.int64) - 1
        subjects = np.loadtxt(base / f"subject_{name}.txt", dtype=np.int64)
        return x, y, subjects

    x_all, y_all, train_subject_ids = split("train")
    x_test, y_test, test_subject_ids = split("test")
    assert x_all.shape == (7352, 128, 9), f"unexpected HAR train shape: {x_all.shape}"
    assert x_test.shape == (2947, 128, 9), f"unexpected HAR test shape: {x_test.shape}"
    train_subjects = np.unique(train_subject_ids)
    assert len(train_subjects) == 21, f"expected 21 training subjects, got {len(train_subjects)}"
    assert len(np.unique(test_subject_ids)) == 9, "expected 9 test subjects"
    assert not (set(train_subjects) & set(np.unique(test_subject_ids))), "train/test subjects overlap"

    val_subjects = np.random.default_rng(seed).choice(train_subjects, 4, replace=False)
    is_val = np.isin(train_subject_ids, val_subjects)
    x_train, y_train = x_all[~is_val], y_all[~is_val]
    x_val, y_val = x_all[is_val], y_all[is_val]

    # Compute channel statistics from training subjects only (all windows and times).
    mean = x_train.mean(axis=(0, 1), keepdims=True)
    std = x_train.std(axis=(0, 1), keepdims=True).clip(1e-6)
    normalize = lambda a: ((a - mean) / std).astype(np.float32)
    return (
        normalize(x_train), y_train, normalize(x_val), y_val,
        normalize(x_test), y_test, val_subjects, mean.reshape(-1), std.reshape(-1),
    )


### 7. Check the HAR data pipeline
Runs the dataset loader before model training and asserts the expected sequence shapes, four validation subjects, valid class IDs, nonempty classes, and valid per-channel statistics.

In [7]:
# HAR download and data-pipeline assertions (runs before model training)
HAR_ROOT = ensure_har_dataset()
xtr, ytr, xva, yva, xte, yte, val_subjects, har_mean, har_std = load_har(HAR_ROOT, SEEDS[0])
assert xtr.shape[1:] == (128, 9) and xva.shape[1:] == (128, 9) and xte.shape == (2947, 128, 9)
assert len(val_subjects) == 4 and np.all((ytr >= 0) & (ytr < 6))
assert np.all(np.bincount(ytr, minlength=6) > 0), "all six classes should occur in the training split"
assert har_mean.shape == (9,) and har_std.shape == (9,) and np.all(har_std > 0)
print("HAR pipeline assertions passed; data root:", HAR_ROOT)
print("shapes:", xtr.shape, xva.shape, xte.shape, "| validation subjects:", val_subjects)


HAR dataset ready at: har/UCI HAR Dataset
HAR pipeline assertions passed; data root: har/UCI HAR Dataset
shapes: (5905, 128, 9) (1447, 128, 9) (2947, 128, 9) | validation subjects: [25 16 29 15]


### 8. Define the HAR class-balance plot
Defines a six-class bar chart for training windows, including walking, upstairs, downstairs, sitting, standing, and laying.

In [8]:
def plot_class_balance(y: np.ndarray, out: str | Path = "figures/har_class_balance.png"):
    names = ["walking", "upstairs", "downstairs", "sitting", "standing", "laying"]
    counts = np.bincount(y, minlength=6)
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(names, counts)
    ax.set_ylabel("Training windows")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout(); fig.savefig(out, dpi=180); plt.close(fig)


### 9. Inspect class counts and save the plot
Prints the training split’s class counts and saves the class-balance figure for the report.

In [9]:
print(np.bincount(ytr, minlength=6))
plot_class_balance(ytr, "figures/har_class_balance.png")

[ 994  860  791 1033 1104 1123]


### 10. Generate the adding-problem data
Implements the assignment’s deterministic generator: two input channels, exactly one marker in each half, and a target equal to the sum of the marked values.

In [10]:
def make_adding(n: int, T: int, seed: int):
    """Fixed assignment generator: random values + one marker in each half."""
    rng = np.random.default_rng(seed)
    x = np.zeros((n, T, 2), dtype=np.float32)
    x[:, :, 0] = rng.random((n, T))
    i1 = rng.integers(0, T // 2, n)
    i2 = rng.integers(T // 2, T, n)
    x[np.arange(n), i1, 1] = 1
    x[np.arange(n), i2, 1] = 1
    y = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]
    return x, y.astype(np.float32)


### 11. Define adding-generator checks
Defines assertions that verify every generated sequence has exactly two markers, one in each half, and that each target matches the marked-value sum.

In [11]:
def check_adding_generator():
    x, y = make_adding(512, 50, 17)
    marker = x[:, :, 1]
    assert np.all(marker.sum(axis=1) == 2)
    assert np.all(marker[:, :25].sum(axis=1) == 1)
    assert np.all(marker[:, 25:].sum(axis=1) == 1)
    assert np.allclose(y, (x[:, :, 0] * marker).sum(axis=1))


### 12. Run the adding-generator checks
Calls the generator test immediately after its definition so data-generation errors are caught before training.

In [12]:
x_check, y_check = make_adding(512, 50, 17)
marker_check = x_check[:, :, 1]
assert np.all(marker_check.sum(axis=1) == 2)
assert np.all(marker_check[:, :25].sum(axis=1) == 1)
assert np.all(marker_check[:, 25:].sum(axis=1) == 1)
assert np.allclose(y_check, (x_check[:, :, 0] * marker_check).sum(axis=1))
print('Adding-problem generator assertions passed.')


Adding-problem generator assertions passed.


### 13. Implement the scratch vanilla RNN cell
Defines a tanh recurrent cell using matrix operations and parameters compatible with PyTorch’s vanilla RNN layer for weight-copy equivalence checks.

In [13]:
class MyRNNCell(nn.Module):
    """Vanilla tanh cell, with PyTorch-compatible parameter names/shapes."""
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True):
        super().__init__()
        self.weight_ih = nn.Parameter(torch.empty(hidden_size, input_size))
        self.weight_hh = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.bias_ih = nn.Parameter(torch.empty(hidden_size)) if bias else None
        self.bias_hh = nn.Parameter(torch.empty(hidden_size)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        bound = self.weight_hh.shape[1] ** -0.5
        for p in self.parameters(): nn.init.uniform_(p, -bound, bound)

    def forward(self, x, h):
        y = x @ self.weight_ih.T + h @ self.weight_hh.T
        if self.bias_ih is not None: y = y + self.bias_ih + self.bias_hh
        return torch.tanh(y)


### 14. Implement the scratch LSTM cell
Defines the input, forget, candidate, and output gate equations using matrix operations, returning the updated hidden and cell states.

In [14]:
class MyLSTMCell(nn.Module):
    """LSTM cell. Gate order follows PyTorch: input, forget, candidate, output."""
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True):
        super().__init__(); self.hidden_size = hidden_size
        self.weight_ih = nn.Parameter(torch.empty(4 * hidden_size, input_size))
        self.weight_hh = nn.Parameter(torch.empty(4 * hidden_size, hidden_size))
        self.bias_ih = nn.Parameter(torch.empty(4 * hidden_size)) if bias else None
        self.bias_hh = nn.Parameter(torch.empty(4 * hidden_size)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        bound = self.hidden_size ** -0.5
        for p in self.parameters(): nn.init.uniform_(p, -bound, bound)

    def forward(self, x, state):
        h, c = state
        # x: (batch,input); h,c: (batch,hidden); gates: (batch,4*hidden)
        gates = x @ self.weight_ih.T + h @ self.weight_hh.T
        if self.bias_ih is not None: gates = gates + self.bias_ih + self.bias_hh
        i, f, g, o = gates.chunk(4, dim=-1)
        i, f, o, g = i.sigmoid(), f.sigmoid(), o.sigmoid(), g.tanh()
        c_new = f * c + i * g
        h_new = o * c_new.tanh()
        return h_new, c_new


### 15. Implement the scratch GRU cell
Defines the reset, update, and candidate gate equations using matrix operations, following PyTorch’s GRU gate convention.

In [15]:
class MyGRUCell(nn.Module):
    """GRU cell with PyTorch's reset/update/new gate convention and equations."""
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True):
        super().__init__(); self.hidden_size = hidden_size
        self.weight_ih = nn.Parameter(torch.empty(3 * hidden_size, input_size))
        self.weight_hh = nn.Parameter(torch.empty(3 * hidden_size, hidden_size))
        self.bias_ih = nn.Parameter(torch.empty(3 * hidden_size)) if bias else None
        self.bias_hh = nn.Parameter(torch.empty(3 * hidden_size)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        bound = self.hidden_size ** -0.5
        for p in self.parameters(): nn.init.uniform_(p, -bound, bound)

    def forward(self, x, h):
        gi = x @ self.weight_ih.T
        gh = h @ self.weight_hh.T
        if self.bias_ih is not None: gi, gh = gi + self.bias_ih, gh + self.bias_hh
        ir, iz, inn = gi.chunk(3, dim=-1)
        hr, hz, hn = gh.chunk(3, dim=-1)
        reset = (ir + hr).sigmoid()
        update = (iz + hz).sigmoid()
        candidate = (inn + reset * hn).tanh()
        return candidate + update * (h - candidate)


### 16. Compare scratch cells with PyTorch
Copies parameters from library recurrent layers into the custom cells, runs a random sequence batch, and computes the maximum output differences.

In [16]:
def check_scratch_cells(batch=5, steps=7, input_size=9, hidden_size=HIDDEN_SIZE):
    """Copy framework weights and assert maximum sequence-output error < 1e-5."""
    torch.manual_seed(2026)
    x = torch.randn(batch, steps, input_size)
    h0 = torch.randn(1, batch, hidden_size)
    c0 = torch.randn(1, batch, hidden_size)
    checks = {}
    for kind, lib, mine in (
        ("RNN", nn.RNN(input_size, hidden_size, batch_first=True), MyRNNCell(input_size, hidden_size)),
        ("LSTM", nn.LSTM(input_size, hidden_size, batch_first=True), MyLSTMCell(input_size, hidden_size)),
        ("GRU", nn.GRU(input_size, hidden_size, batch_first=True), MyGRUCell(input_size, hidden_size)),
    ):
        # nn.RNN/nn.LSTM/nn.GRU suffix single-layer parameters with `_l0`,
        # while the standalone cells use unsuffixed names.
        lib_state = lib.state_dict()
        mine.load_state_dict({
            name: lib_state[f"{name}_l0"].detach().clone()
            for name in ("weight_ih", "weight_hh", "bias_ih", "bias_hh")
        })
        with torch.no_grad():
            if kind == "LSTM": ref, _ = lib(x, (h0, c0))
            else: ref, _ = lib(x, h0)
            h = h0[0]; c = c0[0]; outs=[]
            for t in range(steps):
                if kind == "LSTM": h, c = mine(x[:, t], (h, c))
                else: h = mine(x[:, t], h)
                outs.append(h)
            got = torch.stack(outs, dim=1)
        err = (ref - got).abs().max().item(); checks[kind] = err
        assert err < 1e-5, f"{kind} mismatch: {err}"
    return checks


### 17. Assert scratch-cell equivalence
Checks that the custom RNN, LSTM, and GRU sequence outputs match their PyTorch reference layers within the required tolerance.

In [17]:
scratch_errors = check_scratch_cells()
assert all(error < 1e-5 for error in scratch_errors.values())
print('Scratch-cell max errors:', scratch_errors)


Scratch-cell max errors: {'RNN': 1.1920928955078125e-07, 'LSTM': 1.1920928955078125e-07, 'GRU': 0.0}


### 18. Derive one-bias parameter counts
Computes the hand-count formulas for RNN, LSTM, and GRU cells using input size 9 and the selected hidden size. The assignment’s PyTorch counts add a second bias vector per gate.

In [18]:
def hand_parameter_counts(input_size=9, hidden_size=HIDDEN_SIZE):
    """One-bias hand counts; PyTorch recurrent layers retain two bias vectors."""
    d, h = input_size, hidden_size
    return {"RNN": h*(d+h+1), "LSTM": 4*h*(d+h+1), "GRU": 3*h*(d+h+1)}


### 19. Define HAR recurrent classifiers
Builds the permitted library-backed RNN, LSTM, and GRU HAR classifiers. Each reads a 128-step, 9-channel sequence and classifies from its final hidden state.

In [19]:
class SequenceClassifier(nn.Module):
    """Framework recurrent layer for the permitted HAR model comparisons."""
    def __init__(self, kind: str, input_size=9, hidden_size=HIDDEN_SIZE, classes=6):
        super().__init__()
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]
        kwargs = {"batch_first": True}
        if kind == "RNN":
            kwargs["nonlinearity"] = "tanh"
        self.rnn = layer(input_size, hidden_size, **kwargs)
        self.head = nn.Linear(hidden_size, classes)

    def forward(self, x):
        seq, state = self.rnn(x)
        h = state[0] if isinstance(state, tuple) else state
        return self.head(h[-1])


### 20. Define adding-problem regressors
Builds the RNN, LSTM, and GRU models for the synthetic adding task, predicting one scalar from the final hidden representation.

In [20]:
class AddingRegressor(nn.Module):
    def __init__(self, kind: str, hidden_size=HIDDEN_SIZE):
        super().__init__()
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]
        self.rnn = layer(2, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, state = self.rnn(x)
        h = state[0] if isinstance(state, tuple) else state
        return self.head(h[-1]).squeeze(-1)


### 21. Create data loaders
Wraps NumPy arrays or tensors in PyTorch datasets and batch loaders, with optional shuffling for training.

In [21]:
def _loader(x, y, batch_size=BATCH_SIZE, shuffle=False):
    ds = TensorDataset(torch.as_tensor(x), torch.as_tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


### 22. Train a HAR classifier
Trains one classifier with Adam and cross-entropy, evaluates validation loss each epoch, and restores the checkpoint with the best validation loss.

In [22]:
def train_classifier(model, train_loader, val_loader, epochs=EPOCHS_HAR):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    history, best_state, best_val = [], None, float("inf")
    t0 = time.perf_counter()
    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss_fn(model(x), y).backward()
            opt.step()
        model.eval(); losses = []
        with torch.no_grad():
            for x, y in val_loader:
                losses.append(loss_fn(model(x.to(DEVICE)), y.to(DEVICE)).item())
        val_loss = float(np.mean(losses)); history.append(val_loss)
        if val_loss < best_val:
            best_val, best_state = val_loss, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    sec_per_epoch = (time.perf_counter() - t0) / epochs
    model.load_state_dict(best_state)
    return history, sec_per_epoch


### 23. Evaluate a HAR classifier
Computes test accuracy, macro-F1, and a six-class confusion matrix without updating model parameters.

In [23]:
def evaluate_classifier(model, loader):
    model.eval(); pred, truth = [], []
    with torch.no_grad():
        for x, y in loader:
            pred.extend(model(x.to(DEVICE)).argmax(1).cpu().numpy())
            truth.extend(y.numpy())
    return (accuracy_score(truth, pred), f1_score(truth, pred, average="macro"),
            confusion_matrix(truth, pred, labels=list(range(6))))


### 24. Run the three-model HAR comparison
For each chosen seed, trains and evaluates the RNN, LSTM, and GRU under the shared training budget, then records metrics, parameter counts, timing, and validation-subject IDs.

In [24]:
def run_har(root="har", hidden_size=HIDDEN_SIZE, seeds=SEEDS, epochs=EPOCHS_HAR):
    """Run 3-seed HAR comparisons; writes each completed seed to results.csv."""
    summaries = {}
    for seed in seeds:
        seed_everything(seed)
        xtr, ytr, xv, yv, xt, yt, val_subjects, _, _ = load_har(root, seed)
        if seed == seeds[0]:
            plot_class_balance(ytr)
        tr = _loader(xtr, ytr, shuffle=True)
        va, te = _loader(xv, yv), _loader(xt, yt)
        for kind in ("RNN", "LSTM", "GRU"):
            model = SequenceClassifier(kind, hidden_size=hidden_size)
            history, sec = train_classifier(model, tr, va, epochs)
            acc, f1, cm = evaluate_classifier(model, te)
            row = dict(problem="1-HAR", model=kind, seed=seed, test_acc=acc,
                       macro_f1=f1, params=sum(p.numel() for p in model.parameters()),
                       sec_per_epoch=sec, notes=f"h={hidden_size}; val subjects={list(map(int, val_subjects))}")
            append_result(row); summaries.setdefault(kind, []).append((acc, f1, cm, history))
    return summaries

### 25. Train an adding-problem model
Trains a sequence regressor with mean-squared error and optional gradient clipping, retaining the best validation checkpoint and learning history.

In [25]:
def train_regressor(model, train_loader, val_loader, epochs=EPOCHS_ADDING,
                    grad_clip: float | None = None):
    model.to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss(); history=[]; best_state=None; best=float("inf")
    t0=time.perf_counter()
    for _ in range(epochs):
        model.train()
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True)
            loss_fn(model(x),y).backward()
            if grad_clip is not None: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
        model.eval(); vals=[]
        with torch.no_grad():
            for x,y in val_loader: vals.append(loss_fn(model(x.to(DEVICE)),y.to(DEVICE)).item())
        v=float(np.mean(vals)); history.append(v)
        if v<best:
            best=v; best_state={k:t.detach().cpu().clone() for k,t in model.state_dict().items()}
    sec=(time.perf_counter()-t0)/epochs; model.load_state_dict(best_state)
    return history,sec

### 26. Evaluate adding-problem test MSE
Calculates mean squared error on the held-out adding-problem test set.

In [26]:
@torch.no_grad()
def mse_test(model, loader):
    model.eval(); model.to(DEVICE); pred=[]; true=[]
    for x,y in loader:
        pred.extend(model(x.to(DEVICE)).cpu().numpy()); true.extend(y.numpy())
    return float(np.mean((np.asarray(pred)-np.asarray(true))**2))

### 27. Run adding experiments over sequence lengths
Trains all three recurrent models at lengths 50, 100, 200, and 400 for each seed, records test MSE and timing, and keeps the T=200 curves.

In [27]:
def run_adding(hidden_size=HIDDEN_SIZE, seeds=SEEDS, lengths=(50,100,200,400), epochs=EPOCHS_ADDING):
    """Nine runs per length (3 models x 3 seeds), saving each run as it ends."""
    outputs={}
    for T in lengths:
        outputs[T]={}
        for kind in ("RNN","LSTM","GRU"):
            outputs[T][kind]=[]
            for seed in seeds:
                seed_everything(seed)
                xtr,ytr=make_adding(10_000,T,0)
                xv,yv=make_adding(1_000,T,1)
                xt,yt=make_adding(1_000,T,2)
                tr=_loader(xtr,ytr,shuffle=True); va=_loader(xv,yv); te=_loader(xt,yt)
                model=AddingRegressor(kind,hidden_size)
                curve,sec=train_regressor(model,tr,va,epochs)
                mse=mse_test(model,te)
                append_result(dict(problem="1-adding",model=kind,seed=seed,seq_len=T,
                    test_mse=mse,params=sum(p.numel() for p in model.parameters()),
                    sec_per_epoch=sec,notes=f"h={hidden_size}; solved={mse<0.05}"))
                outputs[T][kind].append((mse,curve))
    return outputs


def plot_adding_curves(results, outdir="figures"):
    out=Path(outdir); out.mkdir(parents=True,exist_ok=True)
    # T=200 validation curves, one line per architecture and seed.
    fig,ax=plt.subplots(figsize=(7,4))
    for kind, runs in results[200].items():
        for i,(_,curve) in enumerate(runs):
            ax.plot(curve, alpha=.45, label=f"{kind} seed {i+1}")
    ax.set(xlabel="Epoch",ylabel="Validation MSE",yscale="log",title="Adding problem, T=200")
    ax.legend(ncol=2,fontsize=8); fig.tight_layout(); fig.savefig(out/"adding_T200_curves.png",dpi=180); plt.close(fig)
    fig,ax=plt.subplots(figsize=(7,4))
    for kind in ("RNN","LSTM","GRU"):
        means=[np.mean([m for m,_ in results[T][kind]]) for T in sorted(results)]
        stds=[np.std([m for m,_ in results[T][kind]],ddof=1) for T in sorted(results)]
        ax.errorbar(sorted(results),means,yerr=stds,marker="o",capsize=3,label=kind)
    ax.axhline(1/6,color="gray",ls="--",label="constant baseline (~0.167)")
    ax.axhline(.05,color="black",ls=":",label="solved threshold")
    ax.set(xlabel="Sequence length T",ylabel="Test MSE",yscale="log")
    ax.legend(); fig.tight_layout(); fig.savefig(out/"adding_mse_vs_length.png",dpi=180); plt.close(fig)


def summarize_adding(results):
    rows=[]
    for T, models in results.items():
        for model, runs in models.items():
            mses=np.array([m for m,_ in runs])
            rows.append(dict(seq_len=T,model=model,mean_test_mse=mses.mean(),
                             sample_std=mses.std(ddof=1),solved_runs=int((mses<.05).sum()),n=len(mses)))
    return pd.DataFrame(rows)


### 28. Run the HAR hidden-size sweep
Trains LSTM and GRU classifiers at hidden sizes 16, 32, 64, and 128 across all three seeds, recording accuracy, macro-F1, parameter counts, and epoch time.

In [28]:
def run_har_hidden_sweep(root="har", sizes=(16,32,64,128), seeds=SEEDS, epochs=EPOCHS_HAR):
    """Required LSTM/GRU hidden-size sweep; saves every run to results.csv."""
    for h in sizes:
        for seed in seeds:
            seed_everything(seed)
            xtr,ytr,xv,yv,xt,yt,_,_,_=load_har(root,seed)
            tr,va,te=_loader(xtr,ytr,shuffle=True),_loader(xv,yv),_loader(xt,yt)
            for kind in ("LSTM","GRU"):
                model=SequenceClassifier(kind,hidden_size=h)
                _,sec=train_classifier(model,tr,va,epochs)
                acc,f1,_=evaluate_classifier(model,te)
                append_result(dict(problem="1-HAR-hidden-sweep",model=f"{kind}-h{h}",seed=seed,
                    test_acc=acc,macro_f1=f1,params=sum(p.numel() for p in model.parameters()),
                    sec_per_epoch=sec,notes=f"hidden_size={h}"))


### 29. Set the LSTM forget-gate bias
Defines the initialization used by the odd-ID ablation, setting the total forget-gate bias to the selected value while accounting for PyTorch’s two bias vectors.

In [29]:
def _init_forget_bias(model: AddingRegressor, value: float):
    """Set total LSTM forget-gate bias to value (PyTorch gate order i,f,g,o)."""
    if not isinstance(model.rnn, nn.LSTM): raise TypeError("Requires LSTM")
    h=model.rnn.hidden_size
    with torch.no_grad():
        model.rnn.bias_ih_l0[h:2*h].fill_(value)
        model.rnn.bias_hh_l0[h:2*h].zero_()


### 30. Run the individual T=200 ablation
Runs the selected even/odd ablation at sequence length 200 over the three seeds and records test results for both settings.

In [30]:
def run_ablation(choice="forget_bias", hidden_size=HIDDEN_SIZE, seeds=SEEDS,
                 epochs=EPOCHS_ADDING):
    """Run one listed T=200 ablation, selected explicitly without a student ID.

    choice='forget_bias': LSTM forget-gate bias 0 versus 1.
    choice='gradient_clip': vanilla RNN, no clipping versus norm-1 clipping.
    """
    if choice == "forget_bias":
        kind="LSTM"; variants=(("forget_bias0",0.0),("forget_bias1",1.0))
    elif choice == "gradient_clip":
        kind="RNN"; variants=(("none",None),("clip1",1.0))
    else:
        raise ValueError("choice must be 'forget_bias' or 'gradient_clip'")
    for seed in seeds:
        xtr,ytr=make_adding(10_000,200,0); xv,yv=make_adding(1_000,200,1); xt,yt=make_adding(1_000,200,2)
        tr,va,te=_loader(xtr,ytr,shuffle=True),_loader(xv,yv),_loader(xt,yt)
        for label,value in variants:
            seed_everything(seed); model=AddingRegressor(kind,hidden_size)
            clip=value if choice=="gradient_clip" and label=="clip1" else None
            if choice=="forget_bias": _init_forget_bias(model,float(value))
            _,sec=train_regressor(model,tr,va,epochs,grad_clip=clip)
            mse=mse_test(model,te)
            append_result(dict(problem="1-adding-ablation",model=f"{kind}-{label}",seed=seed,seq_len=200,
                test_mse=mse,params=sum(p.numel() for p in model.parameters()),sec_per_epoch=sec,
                notes=f"ablation={choice}; solved={mse<.05}"))


### 31. Plot gradients through time
Measures the mean norm of the loss gradient with respect to each recurrent hidden state at T=100, before and after training, and saves the log-scale comparison.

In [31]:
def gradient_norm_curves(seed=SEEDS[0], hidden_size=HIDDEN_SIZE, epochs=EPOCHS_ADDING,
                         out="figures/gradient_norm_T100.png"):
    """Plot ||dL/dh_t|| for each recurrent output, before and after training."""
    T=100; xtr,ytr=make_adding(10_000,T,0); xv,yv=make_adding(1_000,T,1)
    tr,va=_loader(xtr,ytr,shuffle=True),_loader(xv,yv)
    fig,axes=plt.subplots(1,2,figsize=(11,4),sharey=True)
    for ax,trained in zip(axes,(False,True)):
        for kind in ("RNN","LSTM","GRU"):
            seed_everything(seed); model=AddingRegressor(kind).to(DEVICE)
            if trained: train_regressor(model,tr,va,epochs)
            x=torch.as_tensor(xv[:64],device=DEVICE)
            # Unroll the framework cell so each intermediate h_t is accessible.
            cell_cls={"RNN":nn.RNNCell,"LSTM":nn.LSTMCell,"GRU":nn.GRUCell}[kind]
            cell=cell_cls(2,hidden_size).to(DEVICE)
            with torch.no_grad():
                for name in ("weight_ih","weight_hh","bias_ih","bias_hh"):
                    getattr(cell,name).copy_(getattr(model.rnn,f"{name}_l0"))
            h=torch.zeros(x.shape[0],hidden_size,device=DEVICE)
            c=torch.zeros_like(h); states=[]
            for t in range(T):
                if kind=="LSTM": h,c=cell(x[:,t],(h,c))
                else: h=cell(x[:,t],h)
                h.retain_grad(); states.append(h)
            final=states[-1]
            loss=model.head(final).squeeze(-1).sub(torch.as_tensor(yv[:64],device=DEVICE)).square().mean()
            model.zero_grad(set_to_none=True); loss.backward()
            norms=np.array([s.grad.norm(dim=-1).mean().item() for s in states])
            ax.plot(np.arange(T),np.maximum(norms,1e-12),label=kind)
        ax.set_yscale("log"); ax.set_xlabel("Time step t"); ax.set_title("After training" if trained else "Before training")
        ax.grid(True,alpha=.25)
    # Plain text avoids Matplotlib mathtext escaping/version issues.
    axes[0].set_ylabel("Mean gradient norm (L2)"); axes[1].legend()
    fig.tight_layout(); Path(out).parent.mkdir(parents=True,exist_ok=True); fig.savefig(out,dpi=180); plt.close(fig)


### 32. Plot the best HAR confusion matrix
Selects the highest-mean-accuracy architecture and its best seed from the collected test results, then plots its confusion matrix. Mention this selection rule when interpreting the figure.

In [32]:
def plot_har_confusion(summaries, out="figures/har_best_confusion.png"):
    """Choose highest mean-accuracy architecture and plot its best seed matrix."""
    best_kind=max(summaries,key=lambda k:np.mean([r[0] for r in summaries[k]]))
    best=max(summaries[best_kind],key=lambda r:r[0]); cm=best[2]
    fig,ax=plt.subplots(figsize=(6,5)); im=ax.imshow(cm,cmap="Blues")
    names=["walking","upstairs","downstairs","sitting","standing","laying"]
    ax.set(xticks=range(6),yticks=range(6),xticklabels=names,yticklabels=names,
           xlabel="Predicted",ylabel="True",title=f"Best HAR model: {best_kind}")
    plt.setp(ax.get_xticklabels(),rotation=35,ha="right")
    for i in range(6):
        for j in range(6): ax.text(j,i,str(cm[i,j]),ha="center",va="center",color="black")
    fig.colorbar(im,ax=ax); fig.tight_layout(); Path(out).parent.mkdir(parents=True,exist_ok=True); fig.savefig(out,dpi=180); plt.close(fig)

### 33. Summarize HAR results
Builds a table with mean and sample standard deviation of test accuracy and macro-F1, plus parameter count and seconds per epoch for each architecture.

In [33]:
def report_mean_std(summaries):
    """Return per-architecture mean and sample standard deviation."""
    rows=[]
    for kind,runs in summaries.items():
        acc=np.array([x[0] for x in runs]); f1=np.array([x[1] for x in runs])
        rows.append(dict(model=kind,accuracy_mean=acc.mean(),accuracy_sample_std=acc.std(ddof=1),
                         macro_f1_mean=f1.mean(),macro_f1_sample_std=f1.std(ddof=1)))
    return pd.DataFrame(rows)


### 34. Read saved result rows
Loads the shared results CSV into dictionaries so the plotting helpers can summarize the hidden-size sweep and ablation runs.

In [34]:
def read_rows(csv_path: Path):
    with csv_path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def numeric(values):
    return np.asarray([float(v) for v in values if v not in (None, "")], dtype=float)


def mean_std(values):
    a = numeric(values)
    if len(a) == 0:
        return float("nan"), float("nan")
    return float(a.mean()), float(a.std(ddof=1)) if len(a) > 1 else 0.0


def hidden_size_figure(rows, out: Path):
    pattern = re.compile(r"^(LSTM|GRU)-h(\d+)$")
    data = defaultdict(list)
    for row in rows:
        if row.get("problem") != "1-HAR-hidden-sweep":
            continue
        match = pattern.match(row.get("model", ""))
        if match:
            model, h = match.group(1), int(match.group(2))
            data[(model, h)].append(row)
    sizes = sorted({h for _, h in data})
    if not sizes:
        raise ValueError("No 1-HAR-hidden-sweep rows found in results.csv")

    colors = {"LSTM": "#d97706", "GRU": "#15803d"}
    fig, (ax_acc, ax_time) = plt.subplots(1, 2, figsize=(11.5, 4.5))
    for model in ("LSTM", "GRU"):
        acc_mu, acc_sd, sec_mu, sec_sd = [], [], [], []
        for h in sizes:
            runs = data.get((model, h), [])
            m, s = mean_std([r.get("test_acc", "") for r in runs])
            acc_mu.append(100*m); acc_sd.append(100*s)
            m, s = mean_std([r.get("sec_per_epoch", "") for r in runs])
            sec_mu.append(m); sec_sd.append(s)
        ax_acc.errorbar(sizes, acc_mu, yerr=acc_sd, marker="o", capsize=4,
                        linewidth=2, color=colors[model], label=model)
        ax_time.errorbar(sizes, sec_mu, yerr=sec_sd, marker="o", capsize=4,
                         linewidth=2, color=colors[model], label=model)
    ax_acc.set(title="HAR test accuracy", xlabel="Hidden size h", ylabel="Accuracy (%)")
    ax_acc.set_xticks(sizes); ax_acc.grid(True, alpha=.25); ax_acc.legend(frameon=False)
    ax_time.set(title="HAR training time", xlabel="Hidden size h", ylabel="Seconds per epoch")
    ax_time.set_xticks(sizes); ax_time.grid(True, alpha=.25); ax_time.legend(frameon=False)
    fig.suptitle("HAR hidden-size sweep (mean ± sample SD, 3 seeds)", y=1.02, fontsize=13)
    fig.tight_layout()
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=220, bbox_inches="tight")
    plt.close(fig)


def ablation_figure(rows, out: Path):
    data = defaultdict(list)
    for row in rows:
        if row.get("problem") == "1-adding-ablation" and row.get("seq_len") == "200":
            model = row.get("model", "")
            if model in ("LSTM-forget_bias0", "LSTM-forget_bias1"):
                data[model].append(row)
    labels = ["Forget-gate bias 0", "Forget-gate bias 1"]
    keys = ["LSTM-forget_bias0", "LSTM-forget_bias1"]
    if not all(data.get(k) for k in keys):
        raise ValueError("Expected 1-adding-ablation LSTM forget_bias0/1 rows at T=200")
    means, stds = [], []
    for key in keys:
        m, s = mean_std([r.get("test_mse", "") for r in data[key]])
        means.append(m); stds.append(s)

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    colors = ["#4c78a8", "#e45756"]
    bars = ax.bar(labels, means, yerr=stds, capsize=6, color=colors, alpha=.85,
                  edgecolor="black", linewidth=.6, width=.58)
    # Show the individual seed results as well as mean ± sample SD.
    for i, key in enumerate(keys):
        vals = numeric([r.get("test_mse", "") for r in data[key]])
        jitter = np.linspace(-.07, .07, len(vals)) if len(vals) > 1 else [0]
        ax.scatter(np.full(len(vals), i) + jitter, vals, color="black", s=28,
                   zorder=3, label="Individual seed" if i == 0 else None)
        ax.text(i, means[i] + stds[i] + .006, f"mean={means[i]:.4f}\nSD={stds[i]:.4f}",
                ha="center", va="bottom", fontsize=9)
    ax.axhline(1/6, color="#555555", linestyle="--", linewidth=1.5,
               label="Constant baseline (~0.167)")
    ax.axhline(.05, color="#222222", linestyle=":", linewidth=1.5,
               label="Solved threshold (0.05)")
    ax.set_ylim(0, max(.19, max(means[i]+stds[i] for i in range(2))+.035))
    ax.set_ylabel("Test MSE")
    ax.set_title("LSTM forget-gate-bias ablation, adding problem T=200\nMean ± sample SD, 3 seeds")
    ax.grid(axis="y", alpha=.25)
    handles, legend_labels = ax.get_legend_handles_labels()
    fig.tight_layout(rect=(0, .12, 1, 1))
    fig.legend(handles, legend_labels, frameon=False, loc="lower center",
               bbox_to_anchor=(.5, .015), ncol=3)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=220, bbox_inches="tight")
    plt.close(fig)


### 35. Run the full Problem 1 workflow
Coordinates the required cell checks, HAR comparison and sweep, adding experiments, gradient plot, selected ablation, summary tables, and figures.

In [35]:
def run_full_problem1(har_root="har", ablation_choice="forget_bias", hidden_size=HIDDEN_SIZE,
                      seeds=SEEDS, har_epochs=EPOCHS_HAR, adding_epochs=EPOCHS_ADDING):
    """Orchestrate all required Problem 1 experiments and save report tables.

    This is compute-intensive: adding runs alone train 36 models. Use the same
    epoch budgets across architectures and record any reductions in the report.
    """
    print("Scratch-cell equivalence max errors:", check_scratch_cells(hidden_size=hidden_size))
    print("Hand counts (one bias):", hand_parameter_counts(hidden_size=hidden_size))
    har=run_har(har_root,hidden_size,seeds,har_epochs)
    report_mean_std(har).to_csv("har_summary.csv",index=False)
    plot_har_confusion(har)
    run_har_hidden_sweep(har_root,seeds=seeds,epochs=har_epochs)
    adding=run_adding(hidden_size,seeds,epochs=adding_epochs)
    summarize_adding(adding).to_csv("adding_summary.csv",index=False)
    plot_adding_curves(adding)
    gradient_norm_curves(seed=seeds[0],hidden_size=hidden_size,epochs=adding_epochs)
    if ablation_choice is not None:
        run_ablation(ablation_choice,hidden_size=hidden_size,seeds=seeds,epochs=adding_epochs)
        rows = read_rows(RESULTS_PATH)
        hidden_size_figure(rows, Path("figures") / "har_hidden_size_sweep.png")
        ablation_figure(
            rows,
            Path("figures") / "adding_T200_forget_bias_ablation.png",
        )
    print("Saved results.csv and figures. Review the generated summaries before writing the report.")
    return har,adding


### 36. Execute the notebook workflow
Runs the adding-generator check, prints the selected device, and starts all configured Problem 1 experiments with the chosen values.

In [36]:
check_adding_generator()
print("Device:", DEVICE)
run_full_problem1(
    har_root="har",
    ablation_choice=ABLATION_CHOICE,
    hidden_size=HIDDEN_SIZE,
    seeds=SEEDS,
    har_epochs=EPOCHS_HAR,
    adding_epochs=EPOCHS_ADDING,
)

Device: cuda
Scratch-cell equivalence max errors: {'RNN': 1.1920928955078125e-07, 'LSTM': 1.1920928955078125e-07, 'GRU': 0.0}
Hand counts (one bias): {'RNN': 4736, 'LSTM': 18944, 'GRU': 14208}
Saved results.csv and figures. Review the generated summaries before writing the report.


({'RNN': [(0.8150661689854088,
    0.8131887410869335,
    array([[363,  33,  81,  18,   1,   0],
           [ 64, 348,  33,  25,   1,   0],
           [ 58,   0, 359,   3,   0,   0],
           [  3,  23,   0, 416,  49,   0],
           [  1,   7,   0, 115, 409,   0],
           [  0,  30,   0,   0,   0, 507]]),
    [1.284978061914444,
     1.2436179667711258,
     1.0653014381726582,
     1.141112690170606,
     1.0328879728913307,
     0.8454577351609865,
     0.9171815489729246,
     0.8515573913852373,
     0.8799560566743215,
     1.0174851442376773,
     0.8110704024632772,
     0.7149738669395447,
     0.74338440472881,
     0.7231921305259069,
     0.7007449852923552,
     0.7775735532244047,
     0.7325563281774521,
     0.665449163566033,
     0.6383224309732517,
     0.7592870928347111]),
   (0.7074991516796743,
    0.6994140041045117,
    array([[182, 179, 135,   0,   0,   0],
           [ 50, 335,  78,   8,   0,   0],
           [ 63,  62, 295,   0,   0,   0],
           